# Tuần 1 — Mở khoá Tầng 2

Notebook này làm đúng những việc đang chặn tiến độ, theo thứ tự phụ thuộc:

| Bước | Việc | Chi phí |
|---|---|---|
| 1 | Kiểm tra tình trạng toàn bộ run | giây |
| 2 | Dò lệnh `evaluate --split val` trên **một** run | vài phút |
| 3 | Sinh `val_predictions.npz` cho tất cả run đã train | < 1 giờ |
| 4 | Kiểm toàn vẹn npz | giây |
| 5 | **Chẩn đoán `help` / `hurt` — GO/NO-GO của Tầng 2** | phút |
| 6 | Khởi động run Swin-B dò delta | 18 giờ (chạy nền) |

**Nguyên tắc bất di bất dịch:** mọi siêu tham số chọn trên **inner val**. Không bao giờ chạm outer test để chọn ngưỡng hay λ.

Các cell đều **idempotent** — chạy lại không tốn gì, nó tự bỏ qua phần đã xong.

## 0. Thiết lập

Sửa các đường dẫn trong ô này cho khớp máy bạn, rồi chạy.

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np

# ---- Google Drive (bỏ qua nếu chạy local) ----
try:
    from google.colab import drive
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

# ================== SỬA CHO KHỚP MÁY BẠN ==================
REPO   = Path('/content/drive/MyDrive/Sym/Contrastive_V1')   # nơi có train_contrastive.py
RUNS   = Path('/content/drive/MyDrive/Sym/runs_retry1')
DATA   = Path('/content/drive/MyDrive/Sym/data/DISFA')
SPLITS = Path('/content/drive/MyDrive/Sym/splits')
BATCH  = 64

VARIANTS = ['baseline', 'current', 'full']
FOLDS    = [1, 2, 3]
SEEDS    = [0, 1, 2]
AU_NAMES = ['AU1', 'AU2', 'AU4', 'AU6', 'AU9', 'AU12', 'AU25', 'AU26']
# ==========================================================

TRAIN_PY = REPO / 'train_contrastive.py'

def run_logged(cmd, cwd=None):
    """Chạy lệnh, in log theo thời gian thực, raise nếu lỗi."""
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.run([str(c) for c in cmd], cwd=str(cwd or REPO),
                       capture_output=True, text=True)
    if p.stdout: print(p.stdout[-4000:])
    if p.returncode != 0:
        print('--- STDERR ---'); print(p.stderr[-4000:])
        raise RuntimeError(f'lệnh thất bại, mã {p.returncode}')
    print(f'[xong sau {time.time()-t0:.1f}s]\n', flush=True)
    return p

# kiểm tra nhanh
for name, p in [('train_contrastive.py', TRAIN_PY), ('RUNS', RUNS),
                ('DATA', DATA), ('SPLITS', SPLITS)]:
    print(f'{"OK " if p.exists() else "THIẾU"}  {name:22s} {p}')

## 1. Tình trạng hiện tại

Phân biệt ba trạng thái cho mỗi run, vì cách xử lý khác nhau hoàn toàn:

- `..` chưa có `best.pt` → phải train
- `T.` có test nhưng **chưa có val** → chính là tình trạng hiện tại của bạn
- `TV` có cả hai → sẵn sàng cho Tầng 2

In [ ]:
def find_npz(d, kind):
    """Tìm file npz của split `kind` trong thư mục d, chấp nhận vài cách đặt tên."""
    if not d.exists(): return None
    for pat in (f'{kind}_predictions.npz', f'*{kind}*.npz'):
        hits = sorted(d.glob(pat))
        if hits: return hits[0]
    return None

def run_state(variant, fold, seed):
    out = RUNS / variant / f'fold{fold}' / f'seed{seed}'
    return dict(out=out,
                ckpt = out/'best.pt',
                test = find_npz(out/'evaluation',     'test'),
                val  = find_npz(out/'evaluation_val', 'val') or find_npz(out/'evaluation', 'val'))

rows, need_val, need_train = [], [], []
for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v, f, s)
            has_c, has_t, has_v = st['ckpt'].exists(), st['test'] is not None, st['val'] is not None
            rows.append((v, f, s, has_c, has_t, has_v))
            if not has_c:            need_train.append((v, f, s))
            elif not has_v:          need_val.append((v, f, s))

w = max(len(v) for v in VARIANTS) + 2
hdr = f'{"variant":<{w}}{"fold":>5}   ' + '  '.join(f'{"s"+str(s):>4}' for s in SEEDS)
print(hdr); print('-'*len(hdr))
for v in VARIANTS:
    for f in FOLDS:
        cells = []
        for s in SEEDS:
            st = run_state(v, f, s)
            if   st['val'] is not None:   cells.append(f'{"TV":>4}')
            elif st['test'] is not None:  cells.append(f'{"T.":>4}')
            elif st['ckpt'].exists():     cells.append(f'{"..":>4}')
            else:                          cells.append(f'{"--":>4}')
        print(f'{v:<{w}}{f:>5}   ' + '  '.join(cells))

n = len(rows)
print(f'\nTV = đủ val+test    T. = thiếu val    .. = có ckpt chưa evaluate    -- = chưa train')
print(f'\nCó best.pt : {sum(r[3] for r in rows)}/{n}')
print(f'Có test    : {sum(r[4] for r in rows)}/{n}')
print(f'Có val     : {sum(r[5] for r in rows)}/{n}   <-- cần đưa về bằng số "có best.pt"')
print(f'\nCần sinh val : {len(need_val)} run')
print(f'Chưa train   : {len(need_train)} run')

## 2. Dò lệnh trên MỘT run

Chưa chạy cả loạt vội. Hai rủi ro cần loại trước:

1. **`--split` có thể không phải tên tham số đúng** của CLI này.
2. **`evaluate` có thể ghi đè `test_metrics.json`** nếu dùng chung `--output`. File đó là mốc để kiểm `C0`, mất là hỏng một trong bốn kiểm toàn vẹn.

Nên ô này ghi ra thư mục **riêng** `evaluation_val/` và so sánh `evaluation/` trước–sau.

In [ ]:
pv, pf, ps = need_val[0] if need_val else (VARIANTS[0], FOLDS[0], SEEDS[0])
print(f'Dò trên: {pv} / fold{pf} / seed{ps}\n')

st     = run_state(pv, pf, ps)
out    = st['out']
ev_val = out / 'evaluation_val'
before = sorted(p.name for p in (out/'evaluation').glob('*')) if (out/'evaluation').exists() else []

try:
    run_logged([sys.executable, TRAIN_PY, 'evaluate',
                '--split', 'val',
                '--checkpoint', out/'best.pt',
                '--data-root',  DATA,
                '--manifest',   SPLITS/f'fold{pf}.json',
                '--output',     ev_val,
                '--batch-size', BATCH])
except RuntimeError as e:
    print('!! Lệnh thất bại. Rất có thể tên tham số khác.')
    print('   Chạy ô dưới để xem CLI chấp nhận gì, rồi sửa lại lệnh.\n', e)

after = sorted(p.name for p in (out/'evaluation').glob('*')) if (out/'evaluation').exists() else []
print('evaluation_val/ :', sorted(p.name for p in ev_val.glob("*")) if ev_val.exists() else 'KHÔNG TẠO ĐƯỢC')
print('evaluation/ trước:', before)
print('evaluation/ sau  :', after)
print('\n>>> evaluation/ KHÔNG đổi' if before == after else '\n>>> CẢNH BÁO: evaluation/ đã bị thay đổi!')

In [ ]:
# Chạy ô này NẾU ô trên báo lỗi tham số
run_logged([sys.executable, TRAIN_PY, 'evaluate', '--help'])

## 3. Sinh `val_predictions.npz` cho tất cả run

Chỉ chạy sau khi bước 2 xác nhận lệnh đúng và `evaluation/` không bị đụng.

Đây là **suy luận thuần**, không train. Val chỉ có 3 subject nên nhanh hơn test khoảng ba lần.

In [ ]:
t0, done, failed = time.time(), [], []
for (v, f, s) in need_val:
    st = run_state(v, f, s)
    if st['val'] is not None:            # đã có, bỏ qua
        continue
    try:
        run_logged([sys.executable, TRAIN_PY, 'evaluate',
                    '--split', 'val',
                    '--checkpoint', st['out']/'best.pt',
                    '--data-root',  DATA,
                    '--manifest',   SPLITS/f'fold{f}.json',
                    '--output',     st['out']/'evaluation_val',
                    '--batch-size', BATCH])
        done.append((v, f, s))
    except RuntimeError:
        failed.append((v, f, s))

print(f'\nXong {len(done)} run trong {(time.time()-t0)/60:.1f} phút')
if failed:
    print(f'THẤT BẠI {len(failed)}:', failed)

## 4. Kiểm toàn vẹn

Bốn thứ phải đúng trước khi đưa npz vào decoder. Quan trọng nhất là **val và test phải sinh từ cùng một `best.pt`** — giờ chúng được sinh ở hai thời điểm khác nhau nên đây không còn là kiểm tra hình thức.

In [ ]:
def load_npz(p):
    d = np.load(p, allow_pickle=True)
    return {k: d[k] for k in d.files}

problems = []
for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v, f, s)
            if st['val'] is None or st['test'] is None:
                continue
            tag = f'{v}/fold{f}/seed{s}'
            dv, dt = load_npz(st['val']), load_npz(st['test'])

            # (a) khoá bắt buộc
            for need in ('labels', 'probabilities'):
                for nm, d in (('val', dv), ('test', dt)):
                    if need not in d:
                        problems.append(f'{tag}: {nm} thiếu khoá "{need}" (có: {list(d)})')

            if 'labels' in dv and 'probabilities' in dv:
                # (b) kích thước khớp
                if dv['labels'].shape != dv['probabilities'].shape:
                    problems.append(f'{tag}: val labels{dv["labels"].shape} != probs{dv["probabilities"].shape}')
                # (c) xác suất nằm trong [0,1]
                pr = dv['probabilities']
                if pr.min() < -1e-6 or pr.max() > 1+1e-6:
                    problems.append(f'{tag}: val probabilities ngoài [0,1]: [{pr.min():.3f}, {pr.max():.3f}]')
                if pr.min() >= 0.49:
                    problems.append(f'{tag}: val probabilities >= 0.49 — DẤU HIỆU SIGMOID HAI LẦN')

            # (d) val và test KHÔNG được chung subject
            if 'subjects' in dv and 'subjects' in dt:
                ov = set(np.unique(dv['subjects'])) & set(np.unique(dt['subjects']))
                if ov:
                    problems.append(f'{tag}: val và test CHUNG subject {sorted(ov)} — rò rỉ!')

            # (e) npz không được cũ hơn checkpoint
            ck = st['ckpt'].stat().st_mtime
            for nm, p in (('val', st['val']), ('test', st['test'])):
                if p.stat().st_mtime < ck:
                    problems.append(f'{tag}: {nm}.npz CŨ HƠN best.pt — sinh từ checkpoint khác, phải chạy lại')

# in cấu trúc một file để viết bước sau
ex = next((run_state(v,f,s)['val'] for v in VARIANTS for f in FOLDS for s in SEEDS
           if run_state(v,f,s)['val'] is not None), None)
if ex is not None:
    d = load_npz(ex)
    print(f'--- cấu trúc {ex.name} ---')
    for k, a in d.items():
        print(f'  {k:16s} shape={getattr(a,"shape","?")}  dtype={getattr(a,"dtype","?")}')
        if a.dtype.kind in 'US' or a.dtype == object:
            print(f'{"":18s}ví dụ: {a[:2]}')

print('\n' + ('>>> KHÔNG CÓ VẤN ĐỀ' if not problems else f'>>> {len(problems)} VẤN ĐỀ:'))
for p in problems: print('   -', p)

## 5. Chẩn đoán `help` / `hurt` — GO/NO-GO của Tầng 2

Đây là ô quan trọng nhất của cả notebook.

Với mỗi cặp AU, đếm trên inner val:

- **`help`** — detector vi phạm luật, nhãn thật thoả → luật **sửa được** lỗi này
- **`hurt`** — detector vi phạm, nhãn thật cũng vi phạm → ngoại lệ thật, luật sẽ **phá** một dự đoán đúng

Ô này duyệt **toàn bộ** 28 cặp loại trừ và 56 kéo theo, không lọc trước theo lift — vì câu hỏi đang đặt ra là "có chỗ nào cho luật sửa không", chứ chưa phải "luật nào đạt chuẩn thống kê".

**Cách đọc kết quả:** nếu tổng `help` < ~1% số frame thì ràng buộc logic gần như không có việc gì để làm trên tập nhãn đầy đủ. Đó **không phải thất bại** — đó là một kết quả đáng báo cáo, và nó nói rằng giá trị thật của luật nằm ở chế độ ít nhãn.

In [ ]:
import itertools
from collections import defaultdict

TH = 0.5
agg_excl = defaultdict(lambda: dict(help=0, hurt=0, viol=0))
agg_impl = defaultdict(lambda: dict(help=0, hurt=0, viol=0))
per_subject_help = defaultdict(int)
n_frames = 0          # TỔNG frame đã duyệt, cộng cho MỌI run
n_runs   = 0
fr_help  = 0          # frame PHÂN BIỆT có ít nhất 1 luật loại trừ sửa được
fr_hurt  = 0          # frame PHÂN BIỆT có ít nhất 1 luật loại trừ phá hỏng

for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v, f, s)
            if st['val'] is None: continue
            d  = load_npz(st['val'])
            y  = (d['labels'] > 0.5)
            yh = (d['probabilities'] > TH)
            subj = d['subjects'] if 'subjects' in d else np.zeros(len(y), dtype=int)
            n_frames += len(y); n_runs += 1
            any_h = np.zeros(len(y), bool); any_k = np.zeros(len(y), bool)

            for i, j in itertools.combinations(range(len(AU_NAMES)), 2):
                viol = yh[:, i] & yh[:, j]                 # detector bật cả hai
                true_both = y[:, i] & y[:, j]
                h = viol & ~true_both
                k = (AU_NAMES[i], AU_NAMES[j])
                agg_excl[k]['viol'] += int(viol.sum())
                agg_excl[k]['help'] += int(h.sum())
                agg_excl[k]['hurt'] += int((viol & true_both).sum())
                any_h |= h; any_k |= (viol & true_both)
                for sb in np.unique(subj[h]):
                    per_subject_help[str(sb)] += int((subj[h] == sb).sum())

            for i, j in itertools.permutations(range(len(AU_NAMES)), 2):
                viol = yh[:, i] & ~yh[:, j]                # i bật, j tắt
                true_viol = y[:, i] & ~y[:, j]
                k = (AU_NAMES[i], AU_NAMES[j])
                agg_impl[k]['viol'] += int(viol.sum())
                agg_impl[k]['help'] += int((viol & ~true_viol).sum())
                agg_impl[k]['hurt'] += int((viol & true_viol).sum())

            fr_help += int(any_h.sum()); fr_hurt += int(any_k.sum())

def report(agg, title, arrow, topn=12):
    print(f'\n=== {title} ===')
    rank = sorted(agg.items(), key=lambda kv: kv[1]['help'] - kv[1]['hurt'], reverse=True)
    print(f'{"cặp":<16}{"vi phạm":>10}{"help":>9}{"hurt":>9}{"help-hurt":>11}{"net %":>9}')
    print('-'*64)
    for (a, b), c in rank[:topn]:
        net = c['help'] - c['hurt']
        print(f'{a+arrow+b:<16}{c["viol"]:>10}{c["help"]:>9}{c["hurt"]:>9}'
              f'{net:>11}{100.0*net/max(n_frames,1):>8.3f}%')
    pos = sum(1 for _, c in agg.items() if c['help'] - c['hurt'] > 0)
    print(f'... {len(agg)} cặp, {pos} cặp có help > hurt')

report(agg_excl, 'LOẠI TRỪ  (detector bật cả hai)   <-- TÍN HIỆU CHÍNH', ' - ')
report(agg_impl, 'KÉO THEO  (i bật, j tắt)          <-- chỉ có nghĩa SAU khi lọc theo lift', ' -> ')

print(f'\n{"="*64}')
print(f'Đã duyệt {n_runs} run · tổng {n_frames} frame (mọi variant, mọi seed, mọi fold)')
print('Tỉ lệ dưới đây là phần trăm trên TỔNG số frame đã duyệt.\n')

best_e = max((c['help'] - c['hurt'] for c in agg_excl.values()), default=0)
# DÙNG SỐ FRAME PHÂN BIỆT, không cộng qua các cặp (một frame có thể dính nhiều cặp)
r_help = fr_help / max(n_frames, 1)
r_hurt = fr_hurt / max(n_frames, 1)
r_net  = (fr_help - fr_hurt) / max(n_frames, 1)
print(f'Cặp loại trừ tốt nhất        : net = {best_e:>7}')
print(f'Frame có >=1 luật SỬA được   : {fr_help:>7}  ({100*r_help:.3f}%)')
print(f'Frame có >=1 luật PHÁ hỏng   : {fr_hurt:>7}  ({100*r_hurt:.3f}%)')
print(f'Dư địa ròng (phân biệt)      : {fr_help-fr_hurt:>7}  ({100*r_net:.3f}%)')

if r_net < 0.01:
    print('\n>>> NO-GO cho kỳ vọng lớn: tổng dư địa dưới 1% frame.')
    print('    Luật gần như không có việc để sửa trên tập nhãn đầy đủ.')
    print('    Vẫn chạy decoder (rẻ, vài phút) nhưng đừng kỳ vọng quá +0.3 điểm macro-F1,')
    print('    và chuyển trọng tâm đóng góp sang chế độ ít nhãn.')
elif r_net < 0.03:
    print('\n>>> BIÊN GIỚI: dư địa 1-3% frame. Chạy decoder, kỳ vọng +0.2 đến +0.5 điểm.')
else:
    print('\n>>> CÓ CHỖ CHO LUẬT. Dựng bảng luật và chạy decoder.')

if per_subject_help:
    print('\n--- help phân bố theo subject (xem có bị một subject lái không) ---')
    tot = sum(per_subject_help.values())
    for sb, c in sorted(per_subject_help.items(), key=lambda kv: -kv[1])[:8]:
        print(f'  subject {sb:<10} {c:>8}  ({100.0*c/max(tot,1):.1f}%)')
    top = max(per_subject_help.values()) / max(tot, 1)
    if top > 0.35:
        print(f'\n  CẢNH BÁO: một subject chiếm {100*top:.0f}% tổng help — kết luận không ổn định.')

## 6. Dò delta Swin-B — chạy nền

Đây là bước **giảm rủi ro quan trọng nhất của cả kế hoạch 6 tuần**.

Toàn bộ kế hoạch vượt điểm paper đặt cược vào giả định Swin-B cho khoảng **+1.5 điểm**. Nếu giả định đó sai, bạn cần biết ở tuần 1 chứ không phải tuần 5.

Chạy đúng một cấu hình đã có số ResNet-50 để so trực tiếp: `baseline / fold2 / seed0` hiện đạt **65.03** trên test.

Khoảng 18 giờ. Khởi động rồi để đó, làm việc khác song song.

In [ ]:
PROBE = dict(variant='baseline', fold=2, seed=0, arc='swin_transformer_base')
out = RUNS.parent / 'runs_swin_probe' / PROBE['variant'] / f'fold{PROBE["fold"]}' / f'seed{PROBE["seed"]}'
out.mkdir(parents=True, exist_ok=True)

print('So sánh sẽ là:  ResNet-50 baseline/fold2/seed0 = 65.03 test macro-F1')
print(f'Kết quả Swin-B sẽ nằm ở: {out}\n')
print('Lệnh (bỏ comment để chạy — 18 giờ):\n')
cmd = [sys.executable, TRAIN_PY, 'train',
       '--arc', PROBE['arc'],
       '--variant', PROBE['variant'],
       '--fold', PROBE['fold'], '--seed', PROBE['seed'],
       '--data-root', DATA, '--manifest', SPLITS/f'fold{PROBE["fold"]}.json',
       '--output', out, '--batch-size', 32]
print('$', ' '.join(str(c) for c in cmd))
print('\n# run_logged(cmd)')
print('\nLƯU Ý: batch 32 thay vì 64 — Swin-B nặng hơn, dễ hết VRAM.')
print('Kiểm tên tham số --arc / --variant bằng  train_contrastive.py train --help')

## Checklist tuần 1

- [ ] Bước 2 xác nhận `--split val` chạy được và `evaluation/` không bị đụng
- [ ] Bước 3 đưa số run có val **bằng** số run có `best.pt`
- [ ] Bước 4 báo "KHÔNG CÓ VẤN ĐỀ"
- [ ] Bước 5 cho ra con số `help − hurt` → quyết định mức đầu tư cho Tầng 2
- [ ] Bước 6 đã khởi động, có kết quả sau ~18 giờ
- [ ] Đọc `history.json` chẩn đoán `best_epoch ≤ 2` *(chưa có trong notebook này — cần xem cấu trúc file trước)*

Hai thứ cần gửi lại để viết bước tiếp theo: **output bước 4** (cấu trúc npz) và **output bước 5** (bảng help/hurt).